In [81]:
import pandas as pd

df_results = pd.read_json('serp_fc_results.jsonl', lines=True)

In [82]:
# Add content length columns

from pathlib import Path
BASE = Path("../samples/ymyl_29000/res_20250723_n100").resolve()
SCRAPED_CSV = BASE / "_scraped.csv"

df_scraped = pd.read_csv(SCRAPED_CSV)

df_results = df_results.merge(df_scraped[['url', 'content']], on='url', how='left')

df_results['content_char_count'] = df_results['content'].fillna('').str.len()
df_results['content_word_count'] = df_results['content'].fillna('').str.split().str.len()
df_results['content_sentence_count'] = df_results['content'].fillna('').str.count(r'[.!?]') + 1

In [83]:
# Count T/F claims for each document

df_results['true_count'] = df_results['checker_response'].apply(
    lambda x: sum(r.get('classification') == 'True'
                  for r in (x.get('data', {}).get('results', []) if isinstance(x, dict) else []))
)

df_results['false_count'] = df_results['checker_response'].apply(
    lambda x: sum(r.get('classification') == 'False'
                  for r in (x.get('data', {}).get('results', []) if isinstance(x, dict) else []))
)

In [84]:
ai_results = df_results[df_results['ai_class'] == 'AI']
human_results = df_results[df_results['ai_class'] == 'Human']

ai_count = len(ai_results)
human_count = len(human_results)
total_count = len(df_results)

print('AI count:\t', ai_count)
print('Human count:\t', human_count)
print('---')
print('Total count:\t', total_count)

print()
print('AI share:\t', f'{(ai_count / total_count) * 100:.4f}%')

AI count:	 2209
Human count:	 19565
---
Total count:	 21774

AI share:	 10.1451%


In [85]:
# Group by ai_class and aggregate counts
fact_stats = (
    df_results.groupby('ai_class')[['true_count', 'false_count']]
    .agg(['sum', 'mean', 'median'])
)

print(fact_stats)

         true_count                   false_count                 
                sum       mean median         sum      mean median
ai_class                                                          
AI            29135  13.189226   13.0        2495  1.129470    1.0
Human        203086  10.380066   10.0       24037  1.228571    1.0


In [91]:
import numpy as np

def print_val_stats(df, value_fn, title=None):
    """
    Prints min, max, mean, and median of values derived from df using value_fn.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to use.
    value_fn : callable
        A function that takes a row (Series) and returns a numeric value.
    title : str, optional
        Title to print before the stats.
    """
    values = df.apply(value_fn, axis=1)
    values = values.replace([np.inf, -np.inf], np.nan).dropna()  # clean bad values

    if title:
        print(f"=== {title} ===")
    print(f"Min:    {values.min():.8f}")
    print(f"Max:    {values.max():.8f}")
    print(f"Mean:   {values.mean():.8f}")
    print(f"Median: {values.median():.8f}")
    print(f"Count:  {len(values)}\n")


In [96]:
for count_col in ['true_count', 'false_count']:
    for base_col in ['content_char_count', 'content_word_count', 'content_sentence_count']:
        print_val_stats(
            human_results,
            lambda row, c=count_col, b=base_col: row[c] / row[b] if row[b] > 0 else np.nan,
            title=f"{count_col} / {base_col} (Human)"
        )

=== true_count / content_char_count (Human) ===
Min:    0.00000000
Max:    0.04428571
Mean:   0.00232694
Median: 0.00220000
Count:  19565

=== true_count / content_word_count (Human) ===
Min:    0.00000000
Max:    0.34831461
Mean:   0.01508155
Median: 0.01413882
Count:  19565

=== true_count / content_sentence_count (Human) ===
Min:    0.00000000
Max:    74.00000000
Mean:   0.39982766
Median: 0.31250000
Count:  19565

=== false_count / content_char_count (Human) ===
Min:    0.00000000
Max:    0.01960000
Mean:   0.00029982
Median: 0.00020000
Count:  19565

=== false_count / content_word_count (Human) ===
Min:    0.00000000
Max:    0.14020029
Mean:   0.00195870
Median: 0.00123916
Count:  19565

=== false_count / content_sentence_count (Human) ===
Min:    0.00000000
Max:    45.00000000
Mean:   0.07650364
Median: 0.02222222
Count:  19565



In [97]:
for count_col in ['true_count', 'false_count']:
    for base_col in ['content_char_count', 'content_word_count', 'content_sentence_count']:
        print_val_stats(
            ai_results,
            lambda row, c=count_col, b=base_col: row[c] / row[b] if row[b] > 0 else np.nan,
            title=f"{count_col} / {base_col} (AI)"
        )


=== true_count / content_char_count (AI) ===
Min:    0.00000000
Max:    0.00840000
Mean:   0.00268229
Median: 0.00260000
Count:  2209

=== true_count / content_word_count (AI) ===
Min:    0.00000000
Max:    0.05377721
Mean:   0.01746989
Median: 0.01699346
Count:  2209

=== true_count / content_sentence_count (AI) ===
Min:    0.00000000
Max:    1.72727273
Mean:   0.36116574
Median: 0.35294118
Count:  2209

=== false_count / content_char_count (AI) ===
Min:    0.00000000
Max:    0.00360000
Mean:   0.00023118
Median: 0.00020000
Count:  2209

=== false_count / content_word_count (AI) ===
Min:    0.00000000
Max:    0.02931596
Mean:   0.00148950
Median: 0.00122850
Count:  2209

=== false_count / content_sentence_count (AI) ===
Min:    0.00000000
Max:    6.00000000
Mean:   0.03434388
Median: 0.02173913
Count:  2209

